# Hyperparameter tuning -- filter_base sweep (16 / 24 / 32), filtered dataset

The phase-2 config (`code_phase2_harbin_filtered.ipynb`) picked `filter_base=24` by
reasoning (Heilongjiang's more heterogeneous non-forest land cover needs more
capacity than the Amazon's), not by a systematic search -- a real, disclosed gap
in the project (see `poster_notes.md`, "how did we do the hyperparameter tuning").

This is the cheapest real fix: rerun the exact same pipeline (same filtered
dataset, same augmentation, same epochs/steps, same architecture) at three
`filter_base` values -- 16 (the original paper's value), 24 (the value actually
used), and 32 (one step further) -- and compare. No new code paths, just the
existing training loop run three times with one parameter changed.


In [ ]:
import os
# Force legacy Keras 2 behaviour. Current TF ships Keras 3 by default, which
# breaks this codebase's private optimizer-internals usage and the
# unmaintained `segmentation_models` package -- tf_keras is TF's official
# compatibility shim for exactly this situation, same fix already applied
# in predictor.py.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import sys
import numpy as np
import pandas as pd
import PIL
import requests
import tensorflow as tf
import tf_keras as keras
from tf_keras.models import *
from tf_keras.layers import *
from tf_keras.optimizers import *
from tf_keras.losses import *
from tf_keras import backend as K
from tf_keras.callbacks import ModelCheckpoint
from tf_keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import *


## Colab setup -- reuses the same filtered ground-truth archive as `code_phase2_harbin_filtered.ipynb`

In [ ]:
GROUND_TRUTH_VERSION = "filtered"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = "/content/COMP0173_poster_pre"
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/jy-gfm/COMP0173_poster_pre.git {REPO_DIR}
    os.chdir(REPO_DIR)

    TAR_NAME = f"Haerbing_ground_truth_{GROUND_TRUTH_VERSION}"
    DRIVE_TAR_PATH = f"/content/drive/MyDrive/COMP0173/{TAR_NAME}.tar"
    LOCAL_DATA_DIR = f"/content/{TAR_NAME}"

    if not os.path.exists(LOCAL_DATA_DIR):
        extract_root = f"/content/{TAR_NAME}_extract"
        os.makedirs(extract_root, exist_ok=True)
        !tar -xf {DRIVE_TAR_PATH} -C {extract_root}
        LOCAL_DATA_DIR = f"{extract_root}/{TAR_NAME}"

    DATA_DIR = LOCAL_DATA_DIR
else:
    DATA_DIR = f"./Haerbing_ground_truth_{GROUND_TRUTH_VERSION}"

print(f"DATA_DIR = {DATA_DIR}")


## Data ingestion (identical to `code_phase2_harbin_filtered.ipynb`)

In [ ]:
import numpy as np
import glob

def load_split(split):
    image_paths = sorted(glob.glob(f"{DATA_DIR}/{split}/images/*.npy"))
    images, masks = [], []
    for p in image_paths:
        images.append(np.load(p))
        masks.append(np.load(p.replace("/images/", "/masks/")))
    return images, masks

training_images, training_masks = load_split("training")
validation_images_raw, validation_masks_raw = load_split("validation")

validation_images = [im.reshape(1, 512, 512, 4) for im in validation_images_raw]
validation_masks = [m.reshape(1, 512, 512, 1) for m in validation_masks_raw]

print(f"training: {len(training_images)}, validation: {len(validation_images)}")


## Model architecture -- identical to every other notebook in this project (`filter_base` stays a parameter, not hardcoded)

In [ ]:
'''
  Convolutional block with a single conv layer and activation
'''
def convBlock(input, filters, kernel, kernel_init='he_normal', act='relu', transpose=False):
  if transpose == False:
    conv = Conv2D(filters, kernel, padding='same', kernel_initializer=kernel_init)(input)
  else:
    conv = Conv2DTranspose(filters, kernel, padding='same', kernel_initializer=kernel_init)(input)
  conv = Activation(act)(conv)
  return conv

'''
  Convolutional block with two conv layers and two activation layers
'''
def convBlock2(input, filters, kernel, kernel_init='he_normal', act='relu', transpose=False):
  if transpose == False:
    conv = Conv2D(filters, kernel, padding='same', kernel_initializer=kernel_init)(input)
    conv = Activation(act)(conv)
    conv = Conv2D(filters, kernel, padding='same', kernel_initializer=kernel_init)(conv)
    conv = Activation(act)(conv)
  else:
    conv = Conv2DTranspose(filters, kernel, padding='same', kernel_initializer=kernel_init)(input)
    conv = Activation(act)(conv)
    conv = Conv2DTranspose(filters, kernel, padding='same', kernel_initializer=kernel_init)(conv)
    conv = Activation(act)(conv)
  return conv

'''
  Attention gate/block
'''
def attention_block(x, gating, inter_shape, drop_rate=0.25):
    shape_x = K.int_shape(x)
    shape_g = K.int_shape(gating)

    theta_x = Conv2D(inter_shape, kernel_size=1, strides=1, padding='same', kernel_initializer='he_normal', activation=None)(x)
    theta_x = MaxPooling2D((2, 2))(theta_x)
    shape_theta_x = K.int_shape(theta_x)

    phi_g = Conv2D(inter_shape, kernel_size=1, strides=1, padding='same', kernel_initializer='he_normal', activation=None)(gating)

    concat_xg = add([phi_g, theta_x])
    act_xg = Activation('relu')(concat_xg)

    psi = Conv2D(1, kernel_size=1, strides=1, padding='same', kernel_initializer='he_normal', activation=None)(act_xg)
    sigmoid_xg = Activation('sigmoid')(psi)
    shape_sigmoid = K.int_shape(sigmoid_xg)

    upsample_psi = UpSampling2D(interpolation='bilinear', size=(shape_x[1] // shape_sigmoid[1], shape_x[2] // shape_sigmoid[2]))(sigmoid_xg)
    upsample_psi = tf.broadcast_to(upsample_psi, shape=shape_x)
    y = multiply([upsample_psi, x])
    return y

'''
  Attention U-Net model
'''
def UNetAM(trained_weights=None, input_size=(512, 512, 3), drop_rate=0.25, lr=0.0001, filter_base=16):
    inputs = Input(input_size, batch_size=1)

    conv = convBlock2(inputs, filter_base, 3)

    conv0 = MaxPooling2D(pool_size=(2, 2))(conv)
    conv0 = convBlock2(conv0, 2 * filter_base, 3)

    pool0 = MaxPooling2D(pool_size=(2, 2))(conv0)
    conv1 = convBlock2(pool0, 4 * filter_base, 3)

    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    conv2 = convBlock2(pool1, 8 * filter_base, 3)

    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    conv3 = convBlock2(pool2, 16 * filter_base, 3)

    up4 = Conv2DTranspose(8 * filter_base, kernel_size=2, strides=2, kernel_initializer='he_normal')(conv3)
    merge4 = attention_block(conv2, conv3, 8 * filter_base, drop_rate)
    conv4 = concatenate([up4, merge4])
    conv4 = convBlock2(conv4, 8 * filter_base, 3)

    up5 = Conv2DTranspose(4 * filter_base, kernel_size=2, strides=2, kernel_initializer='he_normal')(conv4)
    merge5 = attention_block(conv1, conv4, 4 * filter_base, drop_rate)
    conv5 = concatenate([up5, merge5])
    conv5 = convBlock2(conv5, 4 * filter_base, 3)

    up6 = Conv2DTranspose(2 * filter_base, kernel_size=2, strides=2, kernel_initializer='he_normal')(conv5)
    merge6 = attention_block(conv0, conv5, 2 * filter_base, drop_rate)
    conv6 = concatenate([up6, merge6])
    conv6 = convBlock2(conv6, 2 * filter_base, 3)

    up7 = Conv2DTranspose(1 * filter_base, kernel_size=2, strides=2, kernel_initializer='he_normal')(conv6)
    merge7 = attention_block(conv, conv6, 1 * filter_base, drop_rate)
    conv7 = concatenate([up7, merge7])
    conv7 = concatenate([up7, conv])
    conv7 = convBlock2(conv7, 1 * filter_base, 3)

    out = convBlock(conv7, 1, 1, act='sigmoid')
    model = Model(inputs, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=binary_crossentropy, metrics=['accuracy', 'mse'])

    if trained_weights is not None:
        model.load_weights(trained_weights)
    return model


## Data augmentation generator (identical to `code_phase2_harbin_filtered.ipynb`)

In [ ]:
def adjustData(img, mask, num_class):
    mask[mask > 0.5] = 1  # FOREST
    mask[mask <= 0.5] = 0  # NON-FOREST
    return (img, mask)

def trainGenerator(batch_size, image_array, mask_array, aug_dict,
                    image_save_prefix="image", mask_save_prefix="mask",
                    num_class=2, save_to_dir=None, target_size=(512, 512), seed=1):
    image_datagen = ImageDataGenerator(**aug_dict)
    mask_datagen = ImageDataGenerator(**aug_dict)

    image_generator = image_datagen.flow(image_array, batch_size=batch_size,
                                          save_to_dir=save_to_dir,
                                          save_prefix=image_save_prefix, seed=seed)
    mask_generator = mask_datagen.flow(mask_array, batch_size=batch_size,
                                        save_to_dir=save_to_dir,
                                        save_prefix=mask_save_prefix, seed=seed)

    train_generator = zip(image_generator, mask_generator)
    for (img, mask) in train_generator:
        img, mask = adjustData(img, mask, num_class)
        yield (img, mask)

t_images = np.stack(training_images)
t_masks = np.stack(training_masks)
del training_images, training_masks
import gc
gc.collect()

validation_df = tf.data.Dataset.from_tensor_slices((validation_images, validation_masks))

data_gen_args = dict(rotation_range=180, width_shift_range=0.25, height_shift_range=0.25,
                      shear_range=0.25, zoom_range=0.25, horizontal_flip=True,
                      vertical_flip=True, fill_mode='reflect')


## The sweep -- train filter_base = 16, 24, 32

Everything else (learning rate, epochs, steps/epoch, dataset) is held fixed at the
same confirmed phase-2 config used everywhere else in this project -- only
`filter_base` changes between runs. Each run gets its own saved model/history file
so nothing overwrites the existing headline `unet-attention-4d-harbin-filtered.hdf5`.


In [ ]:
INPUT_SIZE = (512, 512, 4)
LEARNING_RATE = 0.0005
EPOCHS = 20
STEPS_PER_EPOCH = len(t_images)
FILTER_BASE_VALUES = [16, 24, 32]

DRIVE_SAVE_DIR = "/content/drive/MyDrive/COMP0173/" if IN_COLAB else "./"

models = {}
histories = {}

for fb in FILTER_BASE_VALUES:
    print(f"\n=== Training filter_base={fb} ===")
    model = UNetAM(input_size=INPUT_SIZE, lr=LEARNING_RATE, filter_base=fb)
    model_out_path = f"unet-attention-4d-harbin-filtered-fb{fb}.hdf5"
    checkpoint = ModelCheckpoint(model_out_path, monitor='val_accuracy', verbose=1, save_best_only=True)
    train = trainGenerator(1, t_images, t_masks, data_gen_args, save_to_dir=None)

    history = model.fit(train, steps_per_epoch=STEPS_PER_EPOCH, epochs=EPOCHS,
                         validation_data=validation_df, callbacks=[checkpoint])

    np.save(model_out_path.replace('.hdf5', '-history.npy'), history.history)
    if IN_COLAB:
        import shutil
        shutil.copy(model_out_path, DRIVE_SAVE_DIR)
        shutil.copy(model_out_path.replace('.hdf5', '-history.npy'), DRIVE_SAVE_DIR)

    # Reload the best checkpoint (save_best_only) rather than keeping the
    # final-epoch in-memory weights, consistent with every other notebook.
    models[fb] = keras.models.load_model(model_out_path, compile=False)
    histories[fb] = history.history


## Evaluate all three -- same metric functions used everywhere else in this project

In [ ]:
def per_image_accuracy(model, images, masks):
    scores = []
    for img, mask in zip(images, masks):
        pred = np.round(model.predict(img, verbose=0)).flatten()
        scores.append((pred == mask.flatten()).mean())
    return np.array(scores)

def precision_eval(model, image, mask):
  precision = []
  for i in range(len(image)):
      reconstruction = model.predict(image[i]).reshape(mask[i].shape[1], mask[i].shape[2])
      reconstruction = np.round(reconstruction).flatten()
      precision.append(precision_score(mask[i].flatten(), reconstruction, average='weighted'))
  return precision

def recall_eval(model, image, mask):
  recall = []
  for i in range(len(image)):
      reconstruction = model.predict(image[i]).reshape(mask[i].shape[1], mask[i].shape[2])
      reconstruction = np.round(reconstruction).flatten()
      recall.append(recall_score(mask[i].flatten(), reconstruction, average='weighted'))
  return recall

def f1_score_eval_basic(precision, recall):
    prec, rec = np.mean(precision), np.mean(recall)
    return 0 if prec + rec == 0 else 2 * (prec * rec) / (prec + rec)

acc = {}
results = {}
for fb in FILTER_BASE_VALUES:
    acc[fb] = per_image_accuracy(models[fb], validation_images, validation_masks)
    prec = precision_eval(models[fb], validation_images, validation_masks)
    rec = recall_eval(models[fb], validation_images, validation_masks)
    f1 = f1_score_eval_basic(prec, rec)
    results[fb] = {"accuracy": acc[fb].mean(), "precision": np.mean(prec), "recall": np.mean(rec), "f1": f1}
    print(f"filter_base={fb}: accuracy={acc[fb].mean():.4f}  precision={np.mean(prec):.4f}  recall={np.mean(rec):.4f}  f1={f1:.4f}")

import pandas as pd
pd.DataFrame(results).T


## Statistical significance -- paired Wilcoxon between adjacent filter_base values

Same validation images for every run (only `filter_base` differs), so this is a
genuine paired comparison -- same test convention as the filtered-vs-unfiltered
and archfix ablations elsewhere in this project.


In [ ]:
from scipy.stats import wilcoxon

stat_16_24, p_16_24 = wilcoxon(acc[16], acc[24])
stat_24_32, p_24_32 = wilcoxon(acc[24], acc[32])

print(f"filter_base 16 vs 24: statistic={stat_16_24:.2f}, p={p_16_24:.4g}",
      "(significant)" if p_16_24 < 0.05 else "(not significant)")
print(f"filter_base 24 vs 32: statistic={stat_24_32:.2f}, p={p_24_32:.4g}",
      "(significant)" if p_24_32 < 0.05 else "(not significant)")
